# Assignment 03 — Stage 1: SFT (SmolLM2-135M)

This notebook mirrors `train_sft.py` cell by cell.

**Assignment settings**

- 1 epoch over 20,000 SFT pairs
- AdamW, learning rate `5e-5`, batch size `16`, gradient clip `1.0`
- Log every `250` optimizer steps
- Save the final model as `checkpoints/sft/`
- Generate 5 greedy before/after summaries (`max_new_tokens=48`)

**Logging semantics**

`LOG_INTERVAL = 250` means the notebook logs at optimizer steps **250, 500, 750, 1000, 1250**.

- The loss printed at step **250** is the mean of the per-step losses from steps **1–250**.
- The loss printed at step **500** is the mean from steps **251–500**.
- After each log, the training-loss accumulator resets.
- The final `"final"` row in the CSV is the mean over **all 1,250 steps**.

Validation is run only at those five logging points.


## 0. Environment

On Colab, uncomment the clone/install cell below. On a local machine, run the notebook from the repository root with the project virtual environment selected as the kernel.


In [ ]:
# Colab only: uncomment these lines.
# !git clone https://github.com/<your-user>/<your-repo>.git
# %cd <your-repo>
# !pip install -q transformers datasets rouge-score matplotlib


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / "starter").exists(), (
    "Run this notebook from the repository root "
    "(the folder containing starter/, common.py and train_sft.py)."
)
print("Repository root:", ROOT)


## 1. Imports and configuration

In [ ]:
import csv
import time
from functools import partial

import torch
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader

from starter.config import SEED
from starter.models import load_tokenizer, load_causal_lm
from starter.data import load_sft_splits, load_eval_posts
from starter.batching import collate_sft

from common import set_seed, token_logprobs, generate_summary


In [ ]:
BATCH_SIZE = 16
LEARNING_RATE = 5e-5
EPOCHS = 1
GRAD_CLIP = 1.0
LOG_INTERVAL = 250
MAX_NEW_TOKENS = 48

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CHECKPOINT_DIR = ROOT / "checkpoints" / "sft"
LOG_DIR = ROOT / "results" / "training_logs"
SAMPLE_FILE = ROOT / "results" / "samples_sft.md"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_FILE.parent.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Data and DataLoaders

In [ ]:
set_seed(SEED)

tokenizer = load_tokenizer()
train_data, val_data = load_sft_splits(tokenizer)

print("SFT train / validation pairs:", len(train_data), "/", len(val_data))


def print_length_stats(rows, name):
    prompt_lengths = [len(row["prompt_ids"]) for row in rows]
    summary_lengths = [len(row["summary_ids"]) for row in rows]

    print(
        f"{name} prompt tokens  mean/max: "
        f"{sum(prompt_lengths) / len(prompt_lengths):.2f} / {max(prompt_lengths)}"
    )
    print(
        f"{name} summary tokens mean/max: "
        f"{sum(summary_lengths) / len(summary_lengths):.2f} / {max(summary_lengths)}"
        " (includes EOS)"
    )


print_length_stats(train_data, "SFT train")
print_length_stats(val_data, "SFT validation")


In [ ]:
collate_fn = partial(collate_sft, pad_id=tokenizer.pad_token_id)

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=(DEVICE == "cuda"),
)

val_loader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=(DEVICE == "cuda"),
)

print("Train batches per epoch:", len(train_loader))
print("Validation batches:     ", len(val_loader))
print("Optimizer steps:        ", len(train_loader) * EPOCHS)


## 3. Base model and before-SFT summaries

In [ ]:
model = load_causal_lm(device=DEVICE)

eval_examples = load_eval_posts(tokenizer)[:5]

before_outputs = []
for example in eval_examples:
    before_outputs.append(
        generate_summary(
            model,
            tokenizer,
            example["prompt_ids"],
            DEVICE,
            max_new_tokens=MAX_NEW_TOKENS,
        )
    )

for index, (example, output) in enumerate(zip(eval_examples, before_outputs), 1):
    print(f"\nExample {index}")
    print("Reference: ", example["summary"])
    print("Before SFT:", output)


## 4. Loss and validation functions

In [ ]:
def calculate_sft_loss(model, batch, device):
    """Mean negative log-likelihood of the summary tokens only."""
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    prompt_lens = batch["prompt_lens"].to(device)

    log_probs, summary_mask = token_logprobs(
        model,
        input_ids,
        attention_mask,
        prompt_lens,
    )

    selected_log_probs = log_probs[summary_mask]

    if selected_log_probs.numel() == 0:
        return log_probs.sum() * 0.0

    return -selected_log_probs.mean()


In [ ]:
@torch.no_grad()
def evaluate(model, val_loader, device):
    """Token-weighted mean summary NLL over the validation set."""
    was_training = model.training
    model.eval()

    total_nll = 0.0
    total_tokens = 0

    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        prompt_lens = batch["prompt_lens"].to(device)

        log_probs, summary_mask = token_logprobs(
            model,
            input_ids,
            attention_mask,
            prompt_lens,
        )

        selected_log_probs = log_probs[summary_mask]
        total_nll += -selected_log_probs.sum().item()
        total_tokens += selected_log_probs.numel()

    if was_training:
        model.train()

    return total_nll / max(total_tokens, 1)


## 5. Train SFT

This is the long cell. On a Colab T4 it should finish in roughly 30–60 minutes depending on runtime speed.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

log_path = LOG_DIR / "sft_training.csv"

running_loss = 0.0
running_steps = 0
epoch_loss_sum = 0.0
epoch_steps = 0

global_step = 0
start_time = time.time()

model.train()

with open(log_path, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["step", "train_loss", "validation_loss", "elapsed_seconds"])

    for _epoch in range(EPOCHS):
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)

            loss = calculate_sft_loss(model, batch, DEVICE)
            loss.backward()

            clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            loss_value = loss.item()
            running_loss += loss_value
            running_steps += 1
            epoch_loss_sum += loss_value
            epoch_steps += 1
            global_step += 1

            if global_step % LOG_INTERVAL == 0:
                train_loss = running_loss / running_steps
                val_loss = evaluate(model, val_loader, DEVICE)
                elapsed = time.time() - start_time

                writer.writerow([global_step, train_loss, val_loss, elapsed])
                csvfile.flush()

                print(
                    f"Step {global_step} | "
                    f"Train Loss: {train_loss:.4f} | "
                    f"Validation Loss: {val_loss:.4f} | "
                    f"Time: {elapsed / 60:.2f} min"
                )

                running_loss = 0.0
                running_steps = 0

    final_train_loss = epoch_loss_sum / max(epoch_steps, 1)
    final_val_loss = evaluate(model, val_loader, DEVICE)
    elapsed = time.time() - start_time

    writer.writerow(["final", final_train_loss, final_val_loss, elapsed])
    csvfile.flush()

print("\nTraining completed.")
print(f"Final train loss:      {final_train_loss:.4f}")
print(f"Final validation loss: {final_val_loss:.4f}")
print(f"Total time: {elapsed / 60:.2f} minutes")


## 6. Save the SFT checkpoint

In [ ]:
model.save_pretrained(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)
print("Checkpoint saved at:", CHECKPOINT_DIR)


## 7. After-SFT summaries and `results/samples_sft.md`

In [ ]:
with open(SAMPLE_FILE, "w", encoding="utf-8") as file:
    file.write("# SFT Before and After Examples\n\n")

    for index, example in enumerate(eval_examples):
        after = generate_summary(
            model,
            tokenizer,
            example["prompt_ids"],
            DEVICE,
            max_new_tokens=MAX_NEW_TOKENS,
        )

        file.write(f"## Example {index + 1}\n\n")
        file.write(f"**Prompt:**\n\n{example['prompt']}\n\n")
        file.write(f"**Human reference:**\n\n{example['summary']}\n\n")
        file.write(f"**Before SFT:**\n\n{before_outputs[index]}\n\n")
        file.write(f"**After SFT:**\n\n{after}\n\n")
        file.write("---\n\n")

print("Samples saved at:", SAMPLE_FILE)


## 8. Plot the SFT loss (Figure 1)

Creates `results/sft_loss.png` from the CSV log.


In [ ]:
import csv

import matplotlib.pyplot as plt

steps = []
train_losses = []
val_losses = []

with open(log_path, newline="", encoding="utf-8") as file:
    for row in csv.DictReader(file):
        if row["step"] == "final":
            continue
        steps.append(int(row["step"]))
        train_losses.append(float(row["train_loss"]))
        val_losses.append(float(row["validation_loss"]))

plt.figure(figsize=(8, 5))
plt.plot(steps, train_losses, marker="o", label="Train loss")
plt.plot(steps, val_losses, marker="o", label="Validation loss")
plt.xlabel("Optimizer step")
plt.ylabel("Loss")
plt.title("SFT training and validation loss")
plt.legend()
plt.grid(True)
plt.tight_layout()

plot_path = ROOT / "results" / "sft_loss.png"
plt.savefig(plot_path, dpi=150)
plt.show()
print("Plot saved at:", plot_path)


## 9. Optional: copy outputs to Google Drive

Do not commit `checkpoints/` to GitHub. Upload them to Drive or Hugging Face instead.


In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
#
# !mkdir -p /content/drive/MyDrive/a3/results/training_logs
# !cp -r checkpoints/sft /content/drive/MyDrive/a3/
# !cp results/training_logs/sft_training.csv /content/drive/MyDrive/a3/results/training_logs/
# !cp results/samples_sft.md /content/drive/MyDrive/a3/results/


## 10. What you should have after running

```text
checkpoints/sft/
results/training_logs/sft_training.csv
results/samples_sft.md
results/sft_loss.png
```

The CSV should contain rows for steps `250, 500, 750, 1000, 1250` and one final row:

```text
step,train_loss,validation_loss,elapsed_seconds
250,... 
500,...
750,...
1000,...
1250,...
final,...
```

If you only want to run the Task 1 mask check first:

```python
!python test_sft_mask.py
```
